In [1]:
%pip install -q datasets pandas numpy matplotlib

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json
from pathlib import Path

SEED = 345
PILOT_SIZE = 200

# Use only the official training split at this stage.
math_train = load_dataset(
    "EleutherAI/hendrycks_math",
    "algebra",
    split="train"
)

assert len(math_train) >= PILOT_SIZE

# Seeded sampling; retain original row indices.
rng = np.random.default_rng(SEED)
pilot_indices = rng.permutation(len(math_train))[:PILOT_SIZE]

records = []
for position, original_index in enumerate(pilot_indices):
    row = math_train[int(original_index)]

    records.append({
        "question_id": f"math_algebra_train_{original_index}",
        "partition": (
            "router_train" if position < 160
            else "router_validation"
        ),
        "problem": row["problem"],
        "gold_solution": row["solution"],
        "difficulty": row["level"],
        "subject": row["type"],
    })

pilot = pd.DataFrame(records)

# Check that questions are assigned to only one partition.
assert pilot["question_id"].is_unique
assert pilot["partition"].value_counts().to_dict() == {
    "router_train": 160,
    "router_validation": 40,
}

Path("data").mkdir(exist_ok=True)
pilot.to_json(
    "data/pilot_questions.jsonl",
    orient="records",
    lines=True
)

manifest = {
    "dataset": "EleutherAI/hendrycks_math",
    "subset": "algebra",
    "source_split": "train",
    "dataset_fingerprint": math_train._fingerprint,
    "seed": SEED,
    "pilot_size": PILOT_SIZE,
    "original_indices": pilot_indices.tolist(),
}

Path("data/pilot_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

print("Pilot questions:", len(pilot))
print("\nPartitions:")
print(pilot["partition"].value_counts())

print("\nDifficulty distribution:")
print(pilot["difficulty"].value_counts().sort_index())

print("\nExample problem:")
print(pilot.iloc[0]["problem"])

README.md:   0%|          | 0.00/3.93k [00:00<?, ?B/s]

algebra/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  505kB            

algebra/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

algebra/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  353kB            

algebra/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

Pilot questions: 200

Partitions:
partition
router_train         160
router_validation     40
Name: count, dtype: int64

Difficulty distribution:
difficulty
Level 1    13
Level 2    40
Level 3    48
Level 4    43
Level 5    56
Name: count, dtype: int64

Example problem:
The function $\lfloor x\rfloor$ is defined as the largest integer less than or equal to $x$. For example, $\lfloor 5.67\rfloor = 5$, $\lfloor -\tfrac 14\rfloor = -1$, and $\lfloor 8\rfloor = 8$.

What is the range of the function $$f(x) = \lfloor x\rfloor - x~?$$Express your answer in interval notation.


In [3]:
%pip install -q groq

In [6]:
from getpass import getpass
from groq import Groq

client = Groq(
    api_key=getpass("Paste your Groq API key: ").strip(),
    max_retries=0,
    timeout=120.0,
)

SMALL_MODEL = "openai/gpt-oss-20b"
LARGE_MODEL = "openai/gpt-oss-120b"

print("Client created")

Paste your Groq API key: ··········
Client created


In [7]:
import time
from groq import (
    AuthenticationError,
    RateLimitError,
    APIStatusError,
    APIConnectionError,
)

question = (
    "The roots of x^2 - 7x + 10 = 0 are a and b. "
    "Find a^2 + b^2."
)

smoke_results = []

for label, model_id in [
    ("SMALL", SMALL_MODEL),
    ("LARGE", LARGE_MODEL),
]:
    print(f"\n--- {label}: {model_id} ---")
    start = time.perf_counter()

    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Solve the math problem. "
                        "Give a brief explanation and put your final "
                        "answer inside LaTeX \\boxed{...}."
                    ),
                },
                {"role": "user", "content": question},
            ],
            temperature=0,
            max_completion_tokens=2048,
        )

        elapsed = time.perf_counter() - start
        answer = response.choices[0].message.content or ""
        finish = response.choices[0].finish_reason
        usage = response.usage

        smoke_results.append({
            "requested_model": model_id,
            "returned_model": response.model,
            "question": question,
            "response": answer,
            "finish_reason": finish,
            "seconds": elapsed,
            "usage": usage.model_dump() if usage else None,
        })

        print(answer or "[No final answer returned]")
        print(f"\nTime: {elapsed:.2f} seconds")
        print("Finish reason:", finish)

        if usage:
            print("Input tokens:", usage.prompt_tokens)
            print("Output tokens:", usage.completion_tokens)

        if finish == "length":
            print("The response hit the token limit; it may be incomplete.")

    except AuthenticationError:
        print("Key rejected. Rerun Cell 2 with your correct Groq key.")
        break

    except RateLimitError:
        print("Free-tier rate limit reached. Wait before retrying.")
        break

    except APIStatusError as error:
        print(f"API error {error.status_code}: {error.message}")
        break

    except APIConnectionError:
        print("Connection failed. Check connectivity and retry later.")
        break


--- SMALL: openai/gpt-oss-20b ---
For the quadratic \(x^2-7x+10=0\) the sum and product of the roots are  

\[
a+b=7,\qquad ab=10.
\]

Using the identity \(a^2+b^2=(a+b)^2-2ab\),

\[
a^2+b^2 = 7^2-2\cdot10 = 49-20 = 29.
\]

\[
\boxed{29}
\]

Time: 0.44 seconds
Finish reason: stop
Input tokens: 127
Output tokens: 191

--- LARGE: openai/gpt-oss-120b ---
The quadratic  

\[
x^{2}-7x+10=0
\]

has roots \(a\) and \(b\).  
For a quadratic \(x^{2}+px+q=0\),

\[
a+b = -p,\qquad ab = q .
\]

Here \(p=-7\) and \(q=10\), so  

\[
a+b = 7,\qquad ab = 10.
\]

Now  

\[
a^{2}+b^{2} = (a+b)^{2} - 2ab = 7^{2} - 2\cdot 10 = 49 - 20 = 29.
\]

\[
\boxed{29}
\]

Time: 0.80 seconds
Finish reason: stop
Input tokens: 127
Output tokens: 238


In [8]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/COMP345_Adaptive_Routing")
RESULTS_DIR = PROJECT_DIR / "responses"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Results will be saved to:", RESULTS_DIR)

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/COMP345_Adaptive_Routing/responses


In [9]:
import json
import hashlib
import time
from datetime import datetime, timezone
from groq import (
    AuthenticationError,
    RateLimitError,
    APIStatusError,
    APIConnectionError,
)

assert "pilot" in globals(), "Run the MATH dataset-loading cell first."

# Save the selected questions, gold solutions, and split assignments.
pilot.to_json(
    PROJECT_DIR / "pilot_questions.jsonl",
    orient="records",
    lines=True,
)

SYSTEM_PROMPT = (
    "Solve the mathematics problem. "
    "Give a concise explanation and put your final answer "
    "inside LaTeX \\boxed{...}."
)

MODELS = {
    "small": SMALL_MODEL,
    "large": LARGE_MODEL,
}

N_TO_COLLECT = 5  # Check these first before increasing to 200.


def collect_responses(n_questions):
    for _, row in pilot.head(n_questions).iterrows():
        for model_label, model_id in MODELS.items():

            # These settings identify the exact request.
            request = {
                "model": model_id,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row["problem"]},
                ],
                "temperature": 0,
                "max_completion_tokens": 2048,
            }

            request_hash = hashlib.sha256(
                json.dumps(request, sort_keys=True).encode()
            ).hexdigest()[:16]

            output_path = RESULTS_DIR / (
                f"{row['question_id']}_{model_label}_{request_hash}.json"
            )

            if output_path.exists():
                print(f"Already saved: {row['question_id']} / {model_label}")
                continue

            start = time.perf_counter()

            try:
                response = client.chat.completions.create(**request)
            except AuthenticationError:
                print("Authentication failed. Re-enter your key.")
                return
            except RateLimitError:
                print(
                    "Rate limit reached. Completed responses are saved. "
                    "Wait, then rerun this cell to resume."
                )
                return
            except APIStatusError as error:
                print(f"API error {error.status_code}: {error.message}")
                return
            except APIConnectionError:
                print("Connection failed. Rerun this cell later to resume.")
                return

            elapsed = time.perf_counter() - start
            choice = response.choices[0]

            record = {
                "question_id": row["question_id"],
                "partition": row["partition"],
                "difficulty": row["difficulty"],
                "problem": row["problem"],
                "gold_solution": row["gold_solution"],
                "model_label": model_label,
                "requested_model": model_id,
                "returned_model": response.model,
                "request": request,
                "response_text": choice.message.content or "",
                "finish_reason": choice.finish_reason,
                "usage": (
                    response.usage.model_dump()
                    if response.usage else None
                ),
                "latency_seconds": elapsed,
                "collected_at_utc": datetime.now(timezone.utc).isoformat(),
                # Preserve full API output for later auditing.
                "raw_response": response.model_dump(),
            }

            temporary_path = output_path.with_suffix(".tmp")
            temporary_path.write_text(
                json.dumps(record, indent=2),
                encoding="utf-8",
            )
            temporary_path.replace(output_path)

            print(
                f"Saved: {row['question_id']} / {model_label} "
                f"| {elapsed:.2f}s | finish={choice.finish_reason}"
            )

            if choice.finish_reason == "length":
                print(
                    "  NOTE: Token limit reached. "
                    "We need to inspect this response for truncation."
                )

            # Slow collection to reduce free-tier rate-limit errors.
            # Your account's token limits can still require longer waits.
            time.sleep(20)

    print("\nCollection finished.")


collect_responses(N_TO_COLLECT)

Saved: math_algebra_train_786 / small | 0.91s | finish=stop
Saved: math_algebra_train_786 / large | 1.06s | finish=stop
Saved: math_algebra_train_161 / small | 0.66s | finish=stop
Saved: math_algebra_train_161 / large | 1.22s | finish=stop
Saved: math_algebra_train_1690 / small | 1.09s | finish=stop
Saved: math_algebra_train_1690 / large | 1.72s | finish=stop
Saved: math_algebra_train_1647 / small | 0.44s | finish=stop
Saved: math_algebra_train_1647 / large | 0.85s | finish=stop
Saved: math_algebra_train_820 / small | 2.05s | finish=stop
Saved: math_algebra_train_820 / large | 2.14s | finish=stop

Collection finished.


In [10]:
first_id = pilot.iloc[0]["question_id"]

print("PROBLEM")
print(pilot.iloc[0]["problem"])

print("\nDATASET SOLUTION")
print(pilot.iloc[0]["gold_solution"])

for path in sorted(RESULTS_DIR.glob(f"{first_id}_*.json")):
    record = json.loads(path.read_text())

    print("\n" + "=" * 60)
    print(record["model_label"].upper(), record["requested_model"])
    print("Finish reason:", record["finish_reason"])
    print(record["response_text"] or "[No final response]")

PROBLEM
The function $\lfloor x\rfloor$ is defined as the largest integer less than or equal to $x$. For example, $\lfloor 5.67\rfloor = 5$, $\lfloor -\tfrac 14\rfloor = -1$, and $\lfloor 8\rfloor = 8$.

What is the range of the function $$f(x) = \lfloor x\rfloor - x~?$$Express your answer in interval notation.

DATASET SOLUTION
Note that if $0\le x<1$, then $\lfloor x\rfloor = 0$, so $f(x)=-x$. Therefore, the range of $f(x)$ includes the interval $(-1,0]$. This is in fact the whole domain; $f(x)$ can't be less than or equal to $-1$, because $x$ and $\lfloor x\rfloor$ necessarily differ by less than $1$, and $f(x)$ can't be positive, because $\lfloor x\rfloor$ is by definition less than or equal to $x$.

Therefore, the range of $f(x)$ is $\boxed{(-1,0]}$.

LARGE openai/gpt-oss-120b
Finish reason: stop
For any real \(x\),

\[
\lfloor x\rfloor \le x < \lfloor x\rfloor +1 .
\]

Let the fractional part of \(x\) be \(\{x\}=x-\lfloor x\rfloor\).  
Then \(0\le \{x\}<1\) and

\[
f(x)=\lfloor x

In [11]:
%pip install -q "math-verify[antlr4_13_2]"

In [12]:
import re
from math_verify import parse, verify, LatexExtractionConfig


def extract_last_boxed(text):
    """Extract the contents of the last complete \\boxed{...}."""
    matches = list(re.finditer(r"\\boxed\s*\{", text))

    if not matches:
        return None

    start = matches[-1].end()
    depth = 1

    for i in range(start, len(text)):
        # Ignore escaped braces such as \{ and \}.
        backslashes = 0
        j = i - 1
        while j >= 0 and text[j] == "\\":
            backslashes += 1
            j -= 1

        if backslashes % 2:
            continue

        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1

            if depth == 0:
                return text[start:i].strip()

    return None


def grade_response(gold_solution, model_response):
    gold = extract_last_boxed(gold_solution)
    predicted = extract_last_boxed(model_response)

    result = {
        "gold_answer": gold,
        "predicted_answer": predicted,
        "auto_correct": None,
        "grading_status": "needs_review",
    }

    if not gold or not predicted:
        result["grading_status"] = "missing_boxed_answer"
        return result

    try:
        # Parse only the extracted final answers, not intermediate work.
        gold_parsed = parse(
            f"${gold}$",
            extraction_config=[LatexExtractionConfig()],
            fallback_mode="no_fallback",
        )
        predicted_parsed = parse(
            f"${predicted}$",
            extraction_config=[LatexExtractionConfig()],
            fallback_mode="no_fallback",
        )

        if not gold_parsed or not predicted_parsed:
            result["grading_status"] = "parse_failed"
            return result

        correct = bool(verify(gold_parsed, predicted_parsed))

        result["auto_correct"] = correct
        result["grading_status"] = (
            "matched" if correct else "mismatch_review"
        )

    except Exception as error:
        result["grading_status"] = (
            f"grader_error:{type(error).__name__}"
        )

    return result


# Basic checks: spacing, equivalent fractions, and interval endpoints.
assert grade_response(
    r"\boxed{(-1,0]}", r"\boxed{(-1,\,0]}"
)["auto_correct"] is True

assert grade_response(
    r"\boxed{\frac{1}{2}}", r"\boxed{0.5}"
)["auto_correct"] is True

assert grade_response(
    r"\boxed{(-1,0]}", r"\boxed{[-1,0]}"
)["auto_correct"] is False

print("Grader checks passed.")

Grader checks passed.


In [15]:
import json
import hashlib
import pandas as pd
from importlib.metadata import version
from IPython.display import display

graded_rows = []
missing = []

for _, row in pilot.head(50).iterrows():
    for model_label, model_id in MODELS.items():

        request = {
            "model": model_id,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["problem"]},
            ],
            "temperature": 0,
            "max_completion_tokens": 2048,
        }

        request_hash = hashlib.sha256(
            json.dumps(request, sort_keys=True).encode()
        ).hexdigest()[:16]

        path = RESULTS_DIR / (
            f"{row['question_id']}_{model_label}_{request_hash}.json"
        )

        if not path.exists():
            missing.append((row["question_id"], model_label))
            continue

        record = json.loads(path.read_text())
        grade = grade_response(
            record["gold_solution"],
            record["response_text"],
        )

        usage = record.get("usage") or {}

        graded_rows.append({
            "question_id": record["question_id"],
            "model": model_label,
            "difficulty": record["difficulty"],
            **grade,
            "finish_reason": record["finish_reason"],
            "input_tokens": usage.get("prompt_tokens"),
            "output_tokens": usage.get("completion_tokens"),
            "latency_seconds": record["latency_seconds"],
            "response_file": str(path),
        })

graded = pd.DataFrame(graded_rows)

graded.to_csv(
    PROJECT_DIR / "first_five_grading.csv",
    index=False,
)

(PROJECT_DIR / "grader_version.txt").write_text(
    f"math-verify=={version('math-verify')}\n"
)

display(graded[[
    "question_id",
    "model",
    "gold_answer",
    "predicted_answer",
    "auto_correct",
    "grading_status",
    "finish_reason",
]])

if missing:
    print("Missing responses:", missing)

print("\nGrading status counts:")
print(graded.groupby(["model", "grading_status"]).size())

print("\nSaved:", PROJECT_DIR / "first_five_grading.csv")

,question_id,model,gold_answer,predicted_answer,auto_correct,grading_status,finish_reason
0,math_algebra_train_786,small,"(-1,0]","(-1,\,0]",True,matched,stop
1,math_algebra_train_786,large,"(-1,0]","(-1,0]",True,matched,stop
2,math_algebra_train_161,small,-i,-i,True,matched,stop
3,math_algebra_train_161,large,-i,-i,True,matched,stop
4,math_algebra_train_1690,small,18,18,True,matched,stop
...,...,...,...,...,...,...,...
95,math_algebra_train_1446,large,5 + 45i,5+45i,True,matched,stop
96,math_algebra_train_835,small,751,751,True,matched,stop
97,math_algebra_train_835,large,751,751,True,matched,stop
98,math_algebra_train_1712,small,5,5,True,matched,stop



Grading status counts:
model  grading_status      
large  matched                 50
small  matched                 49
       missing_boxed_answer     1
dtype: int64

Saved: /content/drive/MyDrive/COMP345_Adaptive_Routing/first_five_grading.csv


In [14]:
collect_responses(50)

Already saved: math_algebra_train_786 / small
Already saved: math_algebra_train_786 / large
Already saved: math_algebra_train_161 / small
Already saved: math_algebra_train_161 / large
Already saved: math_algebra_train_1690 / small
Already saved: math_algebra_train_1690 / large
Already saved: math_algebra_train_1647 / small
Already saved: math_algebra_train_1647 / large
Already saved: math_algebra_train_820 / small
Already saved: math_algebra_train_820 / large
Saved: math_algebra_train_123 / small | 0.44s | finish=stop
Saved: math_algebra_train_123 / large | 0.66s | finish=stop
Saved: math_algebra_train_370 / small | 0.61s | finish=stop
Saved: math_algebra_train_370 / large | 1.21s | finish=stop
Saved: math_algebra_train_4 / small | 0.51s | finish=stop
Saved: math_algebra_train_4 / large | 1.19s | finish=stop
Saved: math_algebra_train_282 / small | 0.59s | finish=stop
Saved: math_algebra_train_282 / large | 0.63s | finish=stop
Saved: math_algebra_train_964 / small | 0.79s | finish=stop


In [16]:
# Summarize only questions with responses from both models.
paired = graded.pivot(
    index="question_id",
    columns="model",
    values="auto_correct",
).reindex(columns=["small", "large"])

complete_ids = (
    graded.groupby("question_id")["model"].nunique()
    .loc[lambda counts: counts == 2]
    .index
)

paired = paired.loc[paired.index.intersection(complete_ids)]

print("Questions with both responses:", len(paired))
print("Pairs with unresolved grading:", paired.isna().any(axis=1).sum())

if len(paired) == 0:
    print("No complete pairs yet.")

elif paired.isna().any().any():
    print("Review unresolved grading before reporting overall accuracy.")

else:
    small = paired["small"].astype(bool)
    large = paired["large"].astype(bool)

    summary = pd.DataFrame({
        "Metric": [
            "Always-small accuracy",
            "Always-large accuracy",
            "Oracle accuracy",
            "Both correct",
            "Only small correct",
            "Only large correct",
            "Both wrong",
        ],
        "Percentage": [
            100 * small.mean(),
            100 * large.mean(),
            100 * (small | large).mean(),
            100 * (small & large).mean(),
            100 * (small & ~large).mean(),
            100 * (~small & large).mean(),
            100 * (~small & ~large).mean(),
        ],
    })

    display(summary.round(1))
    summary.to_csv(PROJECT_DIR / "baseline_summary.csv", index=False)

Questions with both responses: 50
Pairs with unresolved grading: 1
Review unresolved grading before reporting overall accuracy.


In [17]:
review = graded[
    graded["grading_status"].ne("matched")
    | graded["finish_reason"].ne("stop")
]

for _, row in review.iterrows():
    record = json.loads(Path(row["response_file"]).read_text())

    print("\n" + "=" * 70)
    print("QUESTION ID:", row["question_id"])
    print("MODEL:", row["model"])
    print("GRADING STATUS:", row["grading_status"])
    print("FINISH REASON:", row["finish_reason"])

    print("\nPROBLEM:")
    print(record["problem"])

    print("\nGOLD SOLUTION:")
    print(record["gold_solution"])

    print("\nMODEL RESPONSE:")
    print(record["response_text"] or "[No final response returned]")

    print("\nEXTRACTED GOLD:", row["gold_answer"])
    print("EXTRACTED PREDICTION:", row["predicted_answer"])


QUESTION ID: math_algebra_train_355
MODEL: small
GRADING STATUS: missing_boxed_answer
FINISH REASON: length

PROBLEM:
A portion of the graph of $y = f(x)$ is shown in red below, where $f(x)$ is a quadratic function.  The distance between grid lines is $1$ unit.

What is the sum of all distinct numbers $x$ such that $f(f(f(x)))=-3$ ?

[asy]

size(150);

real ticklen=3;

real tickspace=2;

real ticklength=0.1cm;

real axisarrowsize=0.14cm;

pen axispen=black+1.3bp;

real vectorarrowsize=0.2cm;

real tickdown=-0.5;

real tickdownlength=-0.15inch;

real tickdownbase=0.3;

real wholetickdown=tickdown;

void rr_cartesian_axes(real xleft, real xright, real ybottom, real ytop, real xstep=1, real ystep=1, bool useticks=false, bool complexplane=false, bool usegrid=true) {

import graph;

real i;

if(complexplane) {

label("$\textnormal{Re}$",(xright,0),SE);

label("$\textnormal{Im}$",(0,ytop),NW);

} else {

label("$x$",(xright+0.4,-0.5));

label("$y$",(-0.5,ytop+0.2));

}

ylimits(ybottom,ytop

In [18]:
# Keep the automatic grade unchanged.
# Store our reviewed decision separately.
graded["final_correct"] = graded["auto_correct"].astype("boolean")
graded["review_note"] = ""

mask = (
    (graded["question_id"] == "math_algebra_train_355")
    & (graded["model"] == "small")
)

assert mask.sum() == 1

graded.loc[mask, "final_correct"] = False
graded.loc[mask, "review_note"] = (
    "Completion hit the 2048-token limit. Correct intermediate roots, "
    "but no completed final answer. Counted as unsuccessful under "
    "the fixed generation budget."
)

graded.to_csv(
    PROJECT_DIR / "pilot_grading_reviewed.csv",
    index=False,
)

print("Review saved. Original API response preserved.")

Review saved. Original API response preserved.


In [19]:
paired = graded.pivot(
    index="question_id",
    columns="model",
    values="final_correct",
).reindex(columns=["small", "large"])

assert len(paired) == 50, "Expected 50 questions."
assert not paired.isna().any().any(), "Some grades still need review."

small = paired["small"].astype(bool)
large = paired["large"].astype(bool)

summary = pd.DataFrame({
    "Metric": [
        "Always-small accuracy",
        "Always-large accuracy",
        "Oracle accuracy",
        "Both correct",
        "Only small correct",
        "Only large correct",
        "Both unsuccessful",
    ],
    "Count": [
        small.sum(),
        large.sum(),
        (small | large).sum(),
        (small & large).sum(),
        (small & ~large).sum(),
        (~small & large).sum(),
        (~small & ~large).sum(),
    ],
})

summary["Percentage"] = 100 * summary["Count"] / len(paired)

display(summary)
summary.to_csv(
    PROJECT_DIR / "baseline_summary.csv",
    index=False,
)

,Metric,Count,Percentage
0,Always-small accuracy,49,98.0
1,Always-large accuracy,50,100.0
2,Oracle accuracy,50,100.0
3,Both correct,49,98.0
4,Only small correct,0,0.0
5,Only large correct,1,2.0
6,Both unsuccessful,0,0.0


In [20]:
# Published rates previously checked for these Groq models:
# USD per million input/output tokens.
# These estimate paid deployment cost, not your free-tier bill.
RATES = {
    "small": {"input": 0.075, "output": 0.30},
    "large": {"input": 0.15, "output": 0.60},
}

assert graded["input_tokens"].notna().all()
assert graded["output_tokens"].notna().all()

graded["estimated_cost_usd"] = graded.apply(
    lambda row: (
        row["input_tokens"] * RATES[row["model"]]["input"]
        + row["output_tokens"] * RATES[row["model"]]["output"]
    ) / 1_000_000,
    axis=1,
)

cost_summary = graded.groupby("model").agg(
    questions=("question_id", "count"),
    accuracy=("final_correct", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_latency_seconds=("latency_seconds", "mean"),
    mean_estimated_cost_usd=("estimated_cost_usd", "mean"),
    total_estimated_cost_usd=("estimated_cost_usd", "sum"),
)

cost_summary["accuracy_percent"] = 100 * cost_summary.pop("accuracy")

display(cost_summary.round(6))

small_cost = cost_summary.loc["small", "mean_estimated_cost_usd"]
large_cost = cost_summary.loc["large", "mean_estimated_cost_usd"]

print(
    f"Always-small cost relative to always-large: "
    f"{100 * small_cost / large_cost:.1f}%"
)
print(
    f"Estimated savings from always-small: "
    f"{100 * (1 - small_cost / large_cost):.1f}%"
)

cost_summary.to_csv(PROJECT_DIR / "baseline_cost_summary.csv")
graded.to_csv(PROJECT_DIR / "pilot_grading_reviewed.csv", index=False)

,questions,mean_input_tokens,mean_output_tokens,mean_latency_seconds,mean_estimated_cost_usd,total_estimated_cost_usd,accuracy_percent
model,,,,,,,
large,50,162.98,476.62,1.274719,0.000310,0.015521,100.0
small,50,162.98,577.40,0.855433,0.000185,0.009272,98.0


Always-small cost relative to always-large: 59.7%
Estimated savings from always-small: 40.3%
